In [3]:
from utils import profiler, reader
from typing import List
import tqdm

In [54]:
datafile = "../data/day12_input.txt"
data = reader.read_from_file(datafile)
data = [x.rstrip() for x in data]
data

['...']

# Part 1

### Overview

We need to find the amount of fencing and area of each region. Then multiply them together to find a total cost for that specific region. Finally we must sum all regions' costs.

### Approach
This looks like a job for graph searching algorithms. We can use DFS or BFS. We treat each grid square as a vertex and cardinal directions as edges. Each time we encounter the edge of the board or another letter, we add perimeter. Each time we encounter a new valid square, we add area. We hold a hashset of seen locations so as not to revisit. 


In [25]:
example1 = [
"AAAA",
"BBCD",
"BBCC",
"EEEC"
]

example2 = [
"OOOOO",
"OXOXO",
"OOOOO",
"OXOXO",
"OOOOO"
]

example3 = [
'RRRRIICCFF',
'RRRRIICCCF',
'VVRRRCCFFF',
'VVRCCCJFFF',
'VVVVCJJCFE',
'VVIVCCJJEE',
'VVIIICJJEE',
'MIIIIIJJEE',
'MIIISIJEEE',
'MMMISSJEEE'
]

examples = []
for ex in (example1, example2, example3):
    examples.append([list(x) for x in ex])

examples
    

[[['A', 'A', 'A', 'A'],
  ['B', 'B', 'C', 'D'],
  ['B', 'B', 'C', 'C'],
  ['E', 'E', 'E', 'C']],
 [['O', 'O', 'O', 'O', 'O'],
  ['O', 'X', 'O', 'X', 'O'],
  ['O', 'O', 'O', 'O', 'O'],
  ['O', 'X', 'O', 'X', 'O'],
  ['O', 'O', 'O', 'O', 'O']],
 [['R', 'R', 'R', 'R', 'I', 'I', 'C', 'C', 'F', 'F'],
  ['R', 'R', 'R', 'R', 'I', 'I', 'C', 'C', 'C', 'F'],
  ['V', 'V', 'R', 'R', 'R', 'C', 'C', 'F', 'F', 'F'],
  ['V', 'V', 'R', 'C', 'C', 'C', 'J', 'F', 'F', 'F'],
  ['V', 'V', 'V', 'V', 'C', 'J', 'J', 'C', 'F', 'E'],
  ['V', 'V', 'I', 'V', 'C', 'C', 'J', 'J', 'E', 'E'],
  ['V', 'V', 'I', 'I', 'I', 'C', 'J', 'J', 'E', 'E'],
  ['M', 'I', 'I', 'I', 'I', 'I', 'J', 'J', 'E', 'E'],
  ['M', 'I', 'I', 'I', 'S', 'I', 'J', 'E', 'E', 'E'],
  ['M', 'M', 'M', 'I', 'S', 'S', 'J', 'E', 'E', 'E']]]

In [ ]:
from collections import deque

In [55]:
data_lists = [list(x) for x in data]
@profiler.profile
def part1(grid: List[str]) -> int:
    m = len(grid)
    n = len(grid[0])


    def traverse(i, j):
        """
        Traverse a plot of land starting at i, j using BFS
        """
        seen = set()
        plant_type = grid[i][j]
        q = deque()
        q.append((i, j))

        area = 0
        perimeter = 0

        while q:
            i, j = q.popleft()

            if (i, j) in seen:
                continue
            else:
                seen.add((i, j))
                grid[i][j] = ' '
                area += 1


            neighbors = ((i + 1, j), (i - 1, j), (i, j + 1), (i, j - 1))

            for x, y in neighbors:
                if (x, y) in seen:
                    continue
                elif x < 0 or x >= m or y < 0 or y >= n:
                    perimeter += 1
                elif grid[x][y] != plant_type:
                    perimeter += 1
                else:
                    q.append((x, y))
        return area, perimeter    

    result = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] != ' ':
                area, perimeter = traverse(i, j)
                result += area * perimeter

    return result
    
    
part1(data_lists)



Calling part1: Memory used 0 kB; Execution Time: 0.02503420799621381 s


1467094

# Part 2

### Overview 

Now we need to count the sides rather than the perimeter and multiply by area for the total cost.

### Approach

Counting the number of sides is difficult. We can instead count the number of corners, which must be the same.
Consider a counterclockwise path along the edge of the shape. Each corner has a corresponding adjacent forward edge.
Because the mapping bijective, it proves that the number of corners and sides must be the same.

How do we now count the number of corners?
Corners is are 

In [60]:
data_lists = [list(x) for x in data]

@profiler.profile
def part2(grid: List[List[str]]) -> int:
    # Use graph traversal to explore each one of the regions. 
    # We count area as number of spaces visited.
    # Count the number of sides as the number of corners
    m = len(grid)
    n = len(grid[0])

    def traverse(i, j):
        seen = set()
        q = deque()
        plant_type = grid[i][j]
        q.append((i, j))

        area = 0
        corners = 0

        while q:
            i, j = q.popleft()

            if (i, j) in seen:
                continue
            else:
                seen.add((i, j))
                area += 1
            
            neighbors = ((i + 1, j), (i - 1, j), (i, j + 1), (i, j - 1))
            num_oob = 0
            for x, y in neighbors:
                if (x, y) in seen:
                    continue
                elif x < 0 or x >= m or y < 0 or y >= n:
                    num_oob += 1
                elif grid[x][y] != plant_type:
                    num_oob += 1
                else:
                    q.append((x, y))

            if num_oob == 3:
                corners += 2
            elif num_oob == 2:
                corners += 1

        return area, corners

    
    result = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] != ' ':
                area, perimeter = traverse(i, j)
                result += area * perimeter

    return result

part2(data_lists)

Calling part2: Memory used 16384 kB; Execution Time: 2.3303332079958636 s


66324796

np.int64(126324651851776)

In [58]:
F(2,8,12,8,2) 

np.int64(200385994162176)